# TriageFlow - Week 10 API Notebook
**Applied GenAI & Agentic AI Engineering Course · Week 10**

TriageFlow is a three-agent LangGraph system that routes incoming incidents through a Triage classifier, a Knowledge agent (grounded on runbooks), and an Action agent - with a human approval gate before anything runs. This notebook exercises every endpoint two ways.

Each endpoint is shown two ways:
- **cURL (Windows cmd)** - `%%cmd` cell magic, Windows double-quote syntax
- **Python** - `requests` library, works everywhere

---
### Before you start
1. Server running: `uvicorn app.main:app --reload --port 8000`
2. `.env` filled in with your `OPENAI_API_KEY` (copy `.env.example` → `.env`)
3. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes. Single quotes are not supported in Windows cmd.

In [1]:
# Setup -- run this cell first
import requests, json

BASE = 'http://localhost:8000'

# Knowledge-route demo: a how-to question that routes to the Knowledge agent
DEMO_NOTES = 'How do I restart the payments API?'

# Action-route demo: an imperative request that routes to the Action agent + approval gate
DEMO_ACTION = 'Please restart payments-api now'

DEMO_USER = 'u_1'   # the seeded demo user: prior incidents and prefs exist for this id

# Every endpoint but /health needs a caller. X-User-Id stands in for a verified
# bearer token; in production you replace this one line and the authorization
# checks behind it do not change. Note that the approver is a DIFFERENT person
# from the requester - that is the whole point of a human gate.
AS_USER     = {'X-User-Id': DEMO_USER}
AS_APPROVER = {'X-User-Id': 'ops_lead'}

print('Setup complete.')
print('BASE:', BASE)

Setup complete.
BASE: http://localhost:8000


---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which models are loaded.

> This section is identical across all weeks. Do not modify it.

In [2]:
%%cmd
curl -s http://localhost:8000/health

Microsoft Windows [Version 10.0.26200.8655]
(c) Microsoft Corporation. All rights reserved.

week 10/>curl -s http://localhost:8000/health
{"status":"ok","model":"gpt-5.4-mini-2026-03-17","models":{"triage":"gpt-5.4-nano-2026-03-17","knowledge":"gpt-5.4-mini-2026-03-17"}}
week 10/>

In [3]:
# Health check -- Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

Status : 200
{
  "status": "ok",
  "model": "gpt-5.4-mini-2026-03-17",
  "models": {
    "triage": "gpt-5.4-nano-2026-03-17",
    "knowledge": "gpt-5.4-mini-2026-03-17"
  }
}


---
## 2 · Knowledge Route - `POST /triage`

A how-to question flows: **Triage → Knowledge → Compose → response**.

Key concept: the Knowledge agent retrieves runbook chunks from external memory (pgvector, stubbed here) and answers with inline citation markers (`[doc:0]`). Returns immediately - no approval needed.

Request body:
```json
{ "user_request": "...", "user_id": "..." }
```

Response shape (`completed`):
```json
{
  "status": "completed",
  "thread_id": "th_...",
  "final_answer": "Run kubectl rollout restart... [doc:0]",
  "citations": [{"doc_index": 0, "source": "runbooks/payments-restart.md"}]
}
```

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/triage -H "Content-Type: application/json" -H "X-User-Id: u_1" -d "{\"user_request\": \"How do I restart the payments API?\", \"user_id\": \"u_1\"}"

In [ ]:
# Knowledge route -- Python
r = requests.post(f'{BASE}/triage', json={'user_request': DEMO_NOTES, 'user_id': DEMO_USER}, headers=AS_USER)
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('Status     :', data['status'])        # completed
    print('Thread ID  :', data['thread_id'])
    print('Answer     :', data.get('final_answer'))
    print('Citations  :', json.dumps(data.get('citations'), indent=2))

---
## 3 · Action Route + Approval Gate - `POST /triage` then `POST /approve`

An imperative request flows: **Triage → Action → [PAUSE at approval gate]**.

Key concept: a dynamic `interrupt()` **inside** the `action_execute` node checkpoints state and pauses there, above the node's single mutating call. Nothing has run yet. A reviewer calls `/approve` to either run the action or reject it.

The pause is conditional. `risky(proposal)` decides, so a low-risk proposal such as filing a ticket runs straight through and never reaches this step. A compile-time `interrupt_before` list cannot express that condition, which is why it is not what this graph uses.

Response shape (`pending_approval`):
```json
{
  "status": "pending_approval",
  "thread_id": "th_...",
  "proposed_action": {"tool_name": "restart_service", "arguments": {"service_name": "payments-api"}}
}
```

### Step A - Trigger the action route

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/triage -H "Content-Type: application/json" -H "X-User-Id: u_1" -d "{\"user_request\": \"Please restart payments-api now\", \"user_id\": \"u_1\"}"

In [ ]:
# Action route -- Python: triage pauses at the approval gate
r = requests.post(f'{BASE}/triage', json={'user_request': DEMO_ACTION, 'user_id': DEMO_USER}, headers=AS_USER)
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('Status          :', data['status'])        # pending_approval
    print('Thread ID       :', data['thread_id'])
    print('Proposed action :', json.dumps(data.get('proposed_action'), indent=2))

    thread_id = data['thread_id']   # save for Step B

### Step B - Approve and resume

Pass the `thread_id` from Step A. Set `approved: true` to execute, `false` to reject.

```json
{ "thread_id": "th_...", "approved": true, "reviewer_id": "ops_lead" }
```

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/approve -H "Content-Type: application/json" -H "X-User-Id: ops_lead" -d "{\"thread_id\": \"PASTE_THREAD_ID_HERE\", \"approved\": true, \"reviewer_id\": \"ops_lead\"}"

In [ ]:
# Step B: approve -- run Step A first so thread_id is set
r2 = requests.post(f'{BASE}/approve', json={
    'thread_id': thread_id,
    'approved': True,
    'reviewer_id': 'ops_lead'
}, headers=AS_APPROVER)
if r2.status_code != 200:
    print(f'Error {r2.status_code}:', r2.json())
else:
    data2 = r2.json()
    print('Status       :', data2['status'])      # completed
    print('Final answer :', data2.get('final_answer'))

In [ ]:
# Step B (reject variant) -- run a fresh triage, then reject
r_new = requests.post(f'{BASE}/triage', json={'user_request': DEMO_ACTION, 'user_id': DEMO_USER}, headers=AS_USER)
tid2 = r_new.json()['thread_id']

r_reject = requests.post(f'{BASE}/approve', json={
    'thread_id': tid2,
    'approved': False,
    'reviewer_id': 'ops_lead'
}, headers=AS_APPROVER)
data_rej = r_reject.json()
print('Status       :', data_rej['status'])
print('Final answer :', data_rej.get('final_answer'))  # Action declined by reviewer...

---
## 4 · Failure Mode - Missing Required Field (422)

Pydantic validates the request before any model call is made. Omitting `user_id` returns 422 instantly - no tokens spent.

Key concept: schema validation at the API boundary catches bad input cheaply. `TriageRequest` enforces both `user_request` (min_length=1) and `user_id` (min_length=1) as required fields.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/triage -H "Content-Type: application/json" -H "X-User-Id: u_1" -d "{\"user_request\": \"How do I restart payments?\"}"

In [ ]:
# Failure: missing user_id -- Python
r = requests.post(f'{BASE}/triage', json={'user_request': DEMO_NOTES}, headers=AS_USER)
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects missing user_id, no LLM call made)')
print(json.dumps(r.json(), indent=2))

---
## 5 · Failure Mode - Approve Unknown Thread (409 Conflict)

Calling `/approve` with a `thread_id` not paused at the action gate returns 409. Without this check, the graph would silently accept the state injection on a completed or non-existent thread.

Key concept: the `/approve` endpoint validates `snapshot.next` before updating state. If `action_execute` is not in the next-nodes set, the thread is not awaiting approval.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/approve -H "Content-Type: application/json" -H "X-User-Id: ops_lead" -d "{\"thread_id\": \"th_doesnotexist\", \"approved\": true, \"reviewer_id\": \"ops_lead\"}"

In [ ]:
# Failure: approve unknown thread -- Python
r = requests.post(f'{BASE}/approve', json={
    'thread_id': 'th_doesnotexist',
    'approved': True,
    'reviewer_id': 'ops_lead'
}, headers=AS_APPROVER)
print(f'Status: {r.status_code}  (expected 409 -- thread not paused at action gate)')
print(json.dumps(r.json(), indent=2))

---
## 6 · Failure Mode - Empty User Request (422)

`TriageRequest.user_request` has `min_length=1`. An empty string is caught by Pydantic before the graph runs.

In [ ]:
# Failure: empty user_request -- Python
r = requests.post(f'{BASE}/triage', json={'user_request': '', 'user_id': DEMO_USER}, headers=AS_USER)
print(f'Status: {r.status_code}  (expected 422)')
for err in r.json().get('detail', []):
    print(f'  field={err.get("loc")}  msg={err.get("msg")}')

---

## Deliberate failure modes - the two guards (V3 video)

The V3 video walks these two guards on screen; here we **break them** so you can see the failure each one catches. Both run offline against LangGraph directly - no server, no LLM, no API key.

### Failure 1 - the routing contract

`route_after_triage` must return one of the edge-map keys (`knowledge` / `action` / `compose`). Return anything else and LangGraph raises at the routing step - a **loud crash**, not a silent misroute.

In [ ]:
# Failure 1 - the routing contract (offline; no server, no LLM)
# route_after_triage must return a key that exists in the edge map.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class S(TypedDict, total=False):
    user_request: str
    answer: str

def bad_router(state) -> str:
    return "triage"          # 'triage' is NOT a key in the edge map below

g = StateGraph(S)
g.add_node("triage",  lambda s: s)                  # stub node - no model call needed
g.add_node("compose", lambda s: {"answer": "ok"})
g.add_edge(START, "triage")
g.add_conditional_edges(
    "triage", bad_router,
    {"knowledge": "compose", "action": "compose", "compose": "compose"},   # only 3 keys
)
g.add_edge("compose", END)
graph = g.compile()

try:
    graph.invoke({"user_request": "restart the api"})
    print("no error?!")
except Exception as e:
    print(f"CRASH -> {type(e).__name__}: {e}")
# Lesson: the router's return strings and the edge-map keys must match EXACTLY.
# A mismatch is a loud 500, not a silent misroute - which is the outcome you want.


CRASH -> KeyError: 'triage'


### Failure 2 - the approval gate without its guard

`/approve` first checks the thread is actually **paused at `action_execute`**. Strip that check and approving a *finished* thread silently returns **200 with the old answer** - nothing executed, no error. The guard turns that into a clean **409**.

In [ ]:
# Failure 2 - the approval gate without its snapshot guard (offline)
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class S(TypedDict, total=False):
    kind: str; answer: str


def route(s): return "knowledge" if s.get("kind") == "knowledge" else "execute"


def execute(s):
    # The gate is INSIDE the node and conditional, exactly as app/graph.py does
    # it. A compile-time interrupt_before cannot express the condition.
    decision = interrupt({"about_to_happen": "restart payments-api"})
    if not decision.get("approved"):
        return {"answer": "declined, nothing was done"}
    return {"answer": f"RESTARTED service (approved by {decision.get('reviewer_id')})"}


g = StateGraph(S)
g.add_node("start",     lambda s: s)
g.add_node("knowledge", lambda s: {"answer": "here is the runbook (no action taken)"})
g.add_node("execute",   execute)
g.add_edge(START, "start")
g.add_conditional_edges("start", route, {"knowledge": "knowledge", "execute": "execute"})
g.add_edge("knowledge", END); g.add_edge("execute", END)
graph = g.compile(checkpointer=MemorySaver())

# A knowledge thread finishes immediately - it never pauses at the gate.
cfg = {"configurable": {"thread_id": "kn-1"}}
graph.invoke({"kind": "knowledge"}, config=cfg)
snap = graph.get_state(cfg)
print("knowledge thread snapshot.next:", snap.next, "(empty = finished, NOT paused at the gate)")

# THE GUARD (in app/main.py /approve): require 'action_execute' in snapshot.next, else 409.
print("guard check -> 'execute' in snapshot.next?:", "execute" in (snap.next or ()),
      "=> the real /approve returns 409")

# WITHOUT the guard: resuming a thread that is not paused resumes nothing.
graph.invoke(Command(resume={"approved": True, "reviewer_id": "ops_lead"}), config=cfg)
print("no-guard result (HTTP 200, nothing executed):", graph.get_state(cfg).values["answer"])

# And for contrast, the same graph on an action thread, which DOES pause:
cfg2 = {"configurable": {"thread_id": "ac-1"}}
graph.invoke({"kind": "action"}, config=cfg2)
print("action thread snapshot.next:", graph.get_state(cfg2).next, "(paused at the interrupt)")
graph.invoke(Command(resume={"approved": True, "reviewer_id": "ops_lead"}), config=cfg2)
print("after resume:", graph.get_state(cfg2).values["answer"])
# Lesson: the 3-line snapshot check turns a silent, dangerous no-op into a loud, correct 409.


knowledge thread snapshot.next: () (empty = finished, NOT paused at the gate)
guard check -> 'execute' in snapshot.next?: False => the real /approve returns 409
no-guard result (HTTP 200, nothing executed): here is the runbook (no action taken)
action thread snapshot.next: ('execute',) (paused at the interrupt)
after resume: RESTARTED service (approved by ops_lead)


---
## 7 · Full Raw Response
Dump complete JSON for both endpoints - useful for verifying the schema shape.

In [ ]:
# Full raw response -- knowledge route
r = requests.post(f'{BASE}/triage', json={'user_request': DEMO_NOTES, 'user_id': DEMO_USER}, headers=AS_USER)
print('=== /triage (knowledge) ===')
print(json.dumps(r.json(), indent=2))

In [ ]:
# Full raw response -- action route (paused at gate)
r = requests.post(f'{BASE}/triage', json={'user_request': DEMO_ACTION, 'user_id': DEMO_USER}, headers=AS_USER)
print('=== /triage (action, pending_approval) ===')
print(json.dumps(r.json(), indent=2))

---
## 8 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs - try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [18]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))